# MetroCrowdManager — GRPO on Hugging Face Jobs (A100)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiwakar1997/gluon_openenv/blob/main/notebooks/train_on_hf_jobs.ipynb)

**This notebook is the official re-runnable training entry point** for the
MetroCrowdManager OpenEnv hackathon submission.

It does **not** train the model on the Colab runtime (Colab T4 cannot fit
Gemma-3-27B + GRPO). Instead it submits a real GRPO training job to
**Hugging Face Jobs A100-80GB**, tails the live logs back into the cell
output, embeds the live [Trackio dashboard](https://huggingface.co/spaces/DhiwakarDev/mcm-trackio),
and at end-of-run downloads `rewards.csv` from the Hub repo and renders
loss + reward curves inline.

Pipeline:

```
Colab notebook  ──hf jobs run──▶  HF Jobs A100  ──HTTP──▶  OpenEnv Space
       │                                │                    (rewards)
       │                                │
       │       fetch_job_logs(follow)   │
       │ ◀──────────────────────────────┘
       │
       │       hf_hub_download(rewards.csv, log_history.json)
       │ ◀──── after job ends
       │
       └─▶  matplotlib plots → docs/plots/*.png
```

**Requirements to actually run the training:**
1. A Hugging Face account with [Jobs billing enabled](https://huggingface.co/docs/huggingface_hub/guides/jobs)
   (A100-large is paid).
2. An `HF_TOKEN` with `write` scope, configured in the **Auth** cell below.
3. The OpenEnv Space (`DhiwakarDev/openenv` by default) up and running.

Judges who don't want to spin up a paid run can simply scroll through the
saved cell outputs and click the embedded Trackio dashboard.

## 1. Setup

In [ ]:
# Only upgrade what the notebook actually needs. matplotlib + pandas already ship
# in Colab; we pin pandas <3 so Colab's preinstalled gradio/bqplot/db-dtypes
# (which require pandas <3) keep working alongside trackio.
%pip install -q -U 'huggingface_hub>=0.27' trackio 'pandas<3'

In [ ]:
import os
import time
from pathlib import Path

from huggingface_hub import (
    HfApi,
    fetch_job_logs,
    get_token,
    hf_hub_download,
    inspect_job,
    notebook_login,
    run_job,
)
from IPython.display import IFrame, display, Markdown

## 2. Authenticate with Hugging Face

You need an `HF_TOKEN` with **write** scope. The token must be on an
account that has [Jobs billing enabled](https://huggingface.co/docs/huggingface_hub/guides/jobs).

In [ ]:
# Recent huggingface_hub dropped the `write_permission` kwarg.
# Make sure the token you paste has *write* scope (and Jobs billing enabled
# on the account that owns it) — otherwise `run_job` below will 403.
notebook_login()

# Resolve the token from the cache (~/.huggingface/token) and put it on the
# environment so cell 11 can forward it to the HF Job container as a secret.
HF_TOKEN = get_token() or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found. Re-run notebook_login() and paste a token "
        "that has *write* scope and Jobs billing enabled."
    )
os.environ["HF_TOKEN"] = HF_TOKEN

# Sanity check: confirm the token's account can actually see the gated model.
# If this raises, accept the license at https://huggingface.co/google/gemma-3-27b-it
# on the same account, then re-run this cell.
try:
    HfApi().model_info("google/gemma-3-27b-it", token=HF_TOKEN)
    print("token OK and Gemma-3-27B-it is accessible to this account.")
except Exception as exc:
    print(
        "WARNING: this account cannot access google/gemma-3-27b-it.\n"
        "  -> Open https://huggingface.co/google/gemma-3-27b-it while signed in\n"
        "     as the same HF account, click 'Agree and access', then re-run this cell.\n"
        f"  raw error: {exc}"
    )

## 3. Run parameters

Mirrors `scripts/full_run_hf_job.sh`. The notebook submits **one HF Job per
phase** so the hackathon submission has reward / loss curves for all three
tasks, not just one. Edit `STEPS` / `FLAVOR` / `TIMEOUT` once and they apply
to every phase.

| Knob | What it does | Cost note |
|---|---|---|
| `PHASES` | List of tasks to train (`ticket_booking`, `ticket_issuance`, `crowd_announcement`) | one paid job submitted per entry |
| `STEPS` | GRPO update steps (`max_steps`) per phase. Reward curve is usually informative by ~200. | A100 hours scale linearly |
| `FLAVOR` | HF Jobs hardware. `a100-large` = 1× A100-80GB. | `a100x4` for 4× A100 |
| `TIMEOUT` | Hard kill per job | use `"2h"`+ for full runs |

In [ ]:
PHASES             = ["ticket_booking", "ticket_issuance", "crowd_announcement"]
MODEL              = "google/gemma-3-27b-it"
STEPS              = 10                  # hackathon time budget — was 20; bump for headline runs
FLAVOR             = "a100-large"
TIMEOUT            = "1h"                # per phase; bump alongside STEPS

BATCH_SIZE         = 1
GRAD_ACCUM         = 8
NUM_GENERATIONS    = 2
MAX_COMPLETION_LEN = 512
MAX_SEQ_LEN        = 4096
LR                 = 5e-6
SAVE_STEPS         = 10                  # checkpoint mid-run since STEPS is small
DEBUG_MODE         = 0

REPO_URL           = "https://github.com/Dhiwakar1997/gluon_openenv"
ENV_BASE_URL       = "https://dhiwakardev-openenv.hf.space"
TRACKIO_SPACE_ID   = "DhiwakarDev/mcm-trackio"
TRACKIO_PROJECT    = "mcm-gemma3-27b-full"

# One Hub model repo per phase — adapters + metrics live here.
PUSH_TO_HUB_IDS    = {p: f"DhiwakarDev/mcm-gemma3-27b-grpo-{p}" for p in PHASES}

DOCKER_IMAGE       = "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel"

# Local sink for streamed reward/loss — survives mid-run interrupts so we can
# still plot something even if the HF Job dies before its end-of-run upload.
LOCAL_LOG_DIR      = Path("hf_job_logs")
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"phases={PHASES}\n  model={MODEL}\n  steps/phase={STEPS}  flavor={FLAVOR}  timeout/phase={TIMEOUT}")
for phase, repo in PUSH_TO_HUB_IDS.items():
    print(f"  [{phase}] adapters+metrics -> {repo}")
print(f"  local log dir -> {LOCAL_LOG_DIR.resolve()}")

## 4. Build the in-container training command

`build_payload(phase)` returns the shell one-liner the HF Job container runs
for a single phase. It:
1. Installs deps,
2. Clones this repo,
3. Runs `training/hf_jobs_train_grpo.py` with the per-phase `--push-to-hub-id`
   and `--phase` set correctly.

Each of the three phases gets its own payload (and its own job).

In [ ]:
PIP_DEPS = (
    "'transformers>=4.55,<4.60' 'trl>=1.2,<1.4' 'peft>=0.13,<0.16' "
    "'accelerate>=1.0,<1.5' bitsandbytes datasets trackio 'openenv-core[core]' "
    "hf_transfer pydantic websockets httpx"
)

def build_payload(phase: str) -> str:
    """Shell command for one HF Jobs container, parameterised per phase."""
    train_cmd = (
        f"python training/hf_jobs_train_grpo.py "
        f"--phase {phase} "
        f"--model {MODEL} "
        f"--num-episodes {STEPS} "
        f"--batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --num-generations {NUM_GENERATIONS} "
        f"--max-completion-len {MAX_COMPLETION_LEN} --max-seq-len {MAX_SEQ_LEN} "
        f"--lr {LR} "
        f"--save-steps {SAVE_STEPS} "
        f"--debug-mode {DEBUG_MODE} "
        f"--env-base-url $ENV_BASE_URL "
        f"--trackio-space-id $TRACKIO_SPACE_ID "
        f"--trackio-project {TRACKIO_PROJECT} "
        f"--push-to-hub-id {PUSH_TO_HUB_IDS[phase]} "
        f"--output-dir outputs/full-{phase}"
    )
    return (
        "set -e && "
        "apt-get update -qq && apt-get install -y -qq --no-install-recommends git && "
        f"pip install -q {PIP_DEPS} && "
        f"git clone {REPO_URL} /workspace/Gluon && "
        "cd /workspace/Gluon && "
        f"{train_cmd}"
    )

# Sanity preview of the first phase's payload.
print(f"--- payload preview ({PHASES[0]}) ---")
print(build_payload(PHASES[0])[:600], "...")

## 5. Submit + train — one cell per phase

The next cell defines `stream_phase(phase)` — submits one HF Job, tails its
logs into the cell output, and **persists every line + parsed reward/loss
point to local disk as it arrives**:

```
hf_job_logs/<phase>/
  raw.log       # every streamed log line, append-only, fsync'd
  metrics.csv   # parsed step,reward,loss — flushed per row
  job.json      # job_id + last-known status
```

Why local persistence? `training/hf_jobs_train_grpo.py` only uploads
`rewards.csv` and `log_history.json` to the Hub *after training completes*.
If a job is interrupted, fails, or you Ctrl-C the cell, **the Hub artifacts
never get pushed** — so the only surviving record of the run is what we
captured locally as it streamed.

After the helper, three separate cells (one per phase) call it. Run them
one at a time. To stop a phase cleanly, click the **Stop** button on the
cell — `KeyboardInterrupt` is caught, the partial CSV is preserved, and
the job_id is printed so you can cancel it from the cell at the bottom.

Skipping a phase that already has data is just: don't run that phase's
cell again.

In [ ]:
import json
import re
from datetime import datetime, timezone

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty — re-run the auth cell above before submitting jobs.")

# Initialised once per kernel session; per-phase cells write into it.
try:
    JOBS  # noqa: F821 — preserve any existing entries across re-runs
except NameError:
    JOBS = {}

# Match the highlighted-step line emitted by HighlightRewardLossCallback in
# training/hf_jobs_train_grpo.py — looks like:
#     step=12 >>> reward=0.4321 <<< >>> loss=1.2345 <<<
# Either reward or loss may be absent on a given line; we capture whichever
# is present.
_STEP_RE   = re.compile(r"step=(\d+)")
_REWARD_RE = re.compile(r"reward=([-+]?\d*\.?\d+(?:e[-+]?\d+)?)", re.IGNORECASE)
_LOSS_RE   = re.compile(r"loss=([-+]?\d*\.?\d+(?:e[-+]?\d+)?)",   re.IGNORECASE)


def _submit_phase(phase: str):
    """Submit a single HF Job for `phase`. Returns the job handle."""
    return run_job(
        image=DOCKER_IMAGE,
        command=["bash", "-lc", build_payload(phase)],
        flavor=FLAVOR,
        timeout=TIMEOUT,
        env={
            "ENV_BASE_URL": ENV_BASE_URL,
            "TRACKIO_SPACE_ID": TRACKIO_SPACE_ID,
            "TRL_EXPERIMENTAL_SILENCE": "1",
        },
        # Both env-var names are forwarded — some transformers versions check
        # HUGGING_FACE_HUB_TOKEN, others HF_TOKEN.
        secrets={
            "HF_TOKEN": HF_TOKEN,
            "HUGGING_FACE_HUB_TOKEN": HF_TOKEN,
        },
    )


def stream_phase(phase: str) -> str:
    """Submit one HF Job for `phase`, tail its logs, persist reward+loss
    locally line-by-line. Returns the job_id.

    Safe to Ctrl-C / Stop the cell mid-stream — the partial metrics.csv is
    intact through the last flushed row. The HF Job itself keeps running
    until you cancel it from the cancel-jobs cell at the bottom.
    """
    if phase not in PHASES:
        raise ValueError(f"unknown phase {phase!r}; expected one of {PHASES}")

    phase_dir = LOCAL_LOG_DIR / phase
    phase_dir.mkdir(parents=True, exist_ok=True)
    raw_log_path  = phase_dir / "raw.log"
    metrics_path  = phase_dir / "metrics.csv"
    job_meta_path = phase_dir / "job.json"

    print(f"========== [{phase}] submitting HF Job ==========", flush=True)
    job = _submit_phase(phase)
    JOBS[phase] = job.id

    started_at = datetime.now(timezone.utc).isoformat()
    job_meta_path.write_text(json.dumps({
        "phase": phase,
        "job_id": job.id,
        "started_at": started_at,
        "status_at_exit": "streaming",
    }, indent=2))

    print(f"  job_id={job.id}", flush=True)
    print(f"  track at: https://huggingface.co/jobs/{job.id}", flush=True)
    print(f"  raw log  -> {raw_log_path}", flush=True)
    print(f"  metrics  -> {metrics_path}", flush=True)
    print(f"========== [{phase}] tailing logs (Stop cell to interrupt safely) ==========\n",
          flush=True)

    raw_f = open(raw_log_path, "a", buffering=1)        # line-buffered
    is_new = not metrics_path.exists() or metrics_path.stat().st_size == 0
    metrics_f = open(metrics_path, "a", buffering=1)
    if is_new:
        metrics_f.write("step,reward,loss\n")
        metrics_f.flush()

    status_at_exit = "unknown"
    try:
        for line in fetch_job_logs(job_id=job.id, follow=True):
            print(f"[{phase}] {line}", flush=True)
            raw_f.write(line + "\n")
            raw_f.flush()

            # Only emit a CSV row when the line actually carries a step +
            # at least one of reward/loss — that's the highlighted-callback
            # line from the training script, which is the canonical source.
            m_step = _STEP_RE.search(line)
            if not m_step:
                continue
            m_r = _REWARD_RE.search(line)
            m_l = _LOSS_RE.search(line)
            if not (m_r or m_l):
                continue
            step = m_step.group(1)
            reward = m_r.group(1) if m_r else ""
            loss   = m_l.group(1) if m_l else ""
            metrics_f.write(f"{step},{reward},{loss}\n")
            metrics_f.flush()

        status_at_exit = "log_stream_ended"
    except KeyboardInterrupt:
        status_at_exit = "interrupted"
        print(
            f"\n[!] interrupted while tailing {phase} (job_id={job.id}).\n"
            f"    Partial metrics saved at {metrics_path}.\n"
            f"    The job is still running on HF — run the cancel cell at the\n"
            f"    bottom of the notebook to stop billing.",
            flush=True,
        )
        raise
    except Exception as exc:
        status_at_exit = f"error: {exc!r}"
        print(f"\n[!] log streaming for {phase} ended with error: {exc}", flush=True)
        raise
    finally:
        raw_f.close()
        metrics_f.close()
        # Best-effort final status pull — never block exit on a network call.
        final_status = status_at_exit
        try:
            info = inspect_job(job_id=job.id)
            final_status = getattr(info, "status", None) or status_at_exit
        except Exception:
            pass
        job_meta_path.write_text(json.dumps({
            "phase": phase,
            "job_id": job.id,
            "started_at": started_at,
            "status_at_exit": str(final_status),
        }, indent=2))
        print(f"\n========== [{phase}] cell exit — status={final_status} ==========\n",
              flush=True)

    return job.id


print("stream_phase() ready. Run each phase cell below to submit + tail one job.")

### 5a. Phase 1/3 — `ticket_booking`

Submits one HF Job and tails its logs. Click **Stop** on this cell to
interrupt safely; partial reward/loss survive in `hf_job_logs/ticket_booking/metrics.csv`.

In [ ]:
stream_phase("ticket_booking")

### 5b. Phase 2/3 — `ticket_issuance`

Run after Phase 1 either finishes or you stop it. Same Stop-to-interrupt
behaviour; partial data lives in `hf_job_logs/ticket_issuance/metrics.csv`.

In [ ]:
stream_phase("ticket_issuance")

### 5c. Phase 3/3 — `crowd_announcement`

Last phase. Same Stop-to-interrupt behaviour; partial data lives in
`hf_job_logs/crowd_announcement/metrics.csv`.

In [ ]:
stream_phase("crowd_announcement")

## 6. Per-phase status recap

Logs were already tailed inline by cell 11 as each phase ran. This cell just
prints the **final status** of each job (succeeded / failed / cancelled /
running) so you can sanity-check that every phase actually finished before
moving on to artifact download in section 8.

If a phase shows anything other than `completed` here, the corresponding
`metrics/<phase>/...` artifacts in cell 18 will fail to download — fix the
phase (or shrink `STEPS` / bump `TIMEOUT`) and re-run cell 11 from the top.

In [ ]:
for phase, job_id in JOBS.items():
    info = inspect_job(job_id=job_id)
    status = info.status if hasattr(info, "status") else info
    print(f"  [{phase:>20s}] {job_id} -> {status}")

## 7. Live metrics — Trackio dashboard (embedded)

TRL's `TrackioCallback` streams every step's loss, reward, and per-component
reward breakdown to a public Hugging Face Space. The cell below embeds that
dashboard inline. While the job is running, the curves update live.

In [ ]:
TRACKIO_URL = f"https://{TRACKIO_SPACE_ID.replace('/', '-').lower()}.hf.space"
display(Markdown(f"**Live trackio dashboard:** [{TRACKIO_URL}]({TRACKIO_URL})"))
IFrame(TRACKIO_URL, width=1100, height=720)

## 8. After the jobs end — download metrics + render plots

The patched `training/hf_jobs_train_grpo.py` uploads three artifacts to each
phase's `PUSH_TO_HUB_IDS[phase]` repo at end-of-run:

- the LoRA adapters,
- `metrics/<phase>/rewards.csv`     — every reward score, one row per completion,
- `metrics/<phase>/log_history.json` — TRL trainer log (loss, kl, etc., per step).

We download all three phases here and render the headline plots:

- `docs/plots/reward_curve.png` — reward curves for all phases overlaid,
- `docs/plots/loss_curve.png`   — training loss for all phases overlaid,
- `docs/plots/reward_breakdown.png` — per-component breakdown, one subplot per phase.

In [ ]:
for phase, job_id in JOBS.items():
    info = inspect_job(job_id=job_id)
    status = info.status if hasattr(info, "status") else info
    print(f"  [{phase:>20s}] {job_id} -> {status}")

In [ ]:
metrics_paths = {}
for phase in PHASES:
    repo = PUSH_TO_HUB_IDS[phase]
    try:
        rewards_csv = hf_hub_download(
            repo_id=repo,
            filename=f"metrics/{phase}/rewards.csv",
            repo_type="model",
        )
        log_history_json = hf_hub_download(
            repo_id=repo,
            filename=f"metrics/{phase}/log_history.json",
            repo_type="model",
        )
        metrics_paths[phase] = {"rewards": rewards_csv, "log_history": log_history_json}
        print(f"  [{phase}] rewards={rewards_csv}")
        print(f"  [{phase}] log_history={log_history_json}")
    except Exception as exc:
        print(f"  [{phase}] download failed: {exc}")
        print(f"           (job may still be running, was interrupted, or push_to_hub_id={repo} is empty)")

if not metrics_paths:
    print("\nNo phase produced downloadable Hub artifacts — Hub-sourced plots will be skipped.")
    print("The local-CSV plot cell below will still render whatever was streamed before each job ended.")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

PLOTS_DIR = Path("docs/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Stable colour map shared across all plots (Hub + local-CSV).
PHASE_COLORS = {
    "ticket_booking":     "tab:blue",
    "ticket_issuance":    "tab:orange",
    "crowd_announcement": "tab:green",
}

dfs = {}
log_histories = {}
for phase, paths in metrics_paths.items():
    dfs[phase] = pd.read_csv(paths["rewards"])
    with open(paths["log_history"]) as f:
        log_histories[phase] = json.load(f)
    print(f"  [{phase}] reward rows={len(dfs[phase])}  log entries={len(log_histories[phase])}")

if dfs:
    first_phase = next(iter(dfs))
    print(f"\nfirst rows of {first_phase} rewards.csv:")
    display(dfs[first_phase].head())
else:
    print("\n(No Hub-sourced data loaded — see local-CSV plots below for streamed reward/loss.)")

### 8a. Reward + loss curves from **locally streamed** logs

These plots come from `hf_job_logs/<phase>/metrics.csv` — the rows the
`stream_phase()` cells flushed to disk while logs were arriving. They work
even if a job was interrupted, failed, or never finished pushing artifacts
to the Hub. This is the primary deliverable for the hackathon submission;
the Hub-sourced plots further below are a richer view that only renders
when the jobs ran to completion.

In [ ]:
local_dfs = {}
for phase in PHASES:
    csv_path = LOCAL_LOG_DIR / phase / "metrics.csv"
    if not csv_path.exists():
        print(f"  [{phase}] no local metrics.csv — skipping (phase cell never ran)")
        continue
    df_local = pd.read_csv(csv_path)
    if df_local.empty:
        print(f"  [{phase}] metrics.csv exists but is empty — skipping")
        continue
    df_local = df_local.sort_values("step")
    local_dfs[phase] = df_local
    n_r = df_local["reward"].notna().sum()
    n_l = df_local["loss"].notna().sum()
    print(f"  [{phase}] {len(df_local)} rows ({n_r} with reward, {n_l} with loss) <- {csv_path}")

if not local_dfs:
    print("\nNo locally streamed data yet. Run a Phase 5a/5b/5c cell above first.")
else:
    # Reward — one figure, all phases overlaid.
    fig, ax = plt.subplots(figsize=(11, 5))
    plotted_any = False
    final_means = {}
    for phase, df_local in local_dfs.items():
        sub = df_local.dropna(subset=["reward"])
        if sub.empty:
            continue
        per_step = sub.groupby("step", as_index=False)["reward"].mean().sort_values("step")
        ax.plot(
            per_step["step"], per_step["reward"],
            label=phase,
            color=PHASE_COLORS.get(phase),
            linewidth=2.0,
            marker="o",
            markersize=4,
        )
        final_means[phase] = float(per_step["reward"].iloc[-1])
        plotted_any = True
    if plotted_any:
        ax.set_xlabel("Training step")
        ax.set_ylabel("Reward (streamed)")
        ax.set_title(f"GRPO reward — streamed from job logs ({MODEL})")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / "reward_curve_local.png", dpi=140, bbox_inches="tight")
        plt.show()
        print("\nFinal-step reward (streamed):")
        for phase in PHASES:
            if phase in final_means:
                print(f"  {phase:>20s}: {final_means[phase]:.3f}")
    else:
        plt.close(fig)
        print("No reward values parsed yet — wait for a job to log its first highlighted step.")

    # Loss — one figure, all phases overlaid.
    fig, ax = plt.subplots(figsize=(11, 5))
    plotted_any = False
    for phase, df_local in local_dfs.items():
        sub = df_local.dropna(subset=["loss"])
        if sub.empty:
            continue
        per_step = sub.groupby("step", as_index=False)["loss"].mean().sort_values("step")
        ax.plot(
            per_step["step"], per_step["loss"],
            label=phase,
            color=PHASE_COLORS.get(phase),
            linewidth=2.0,
            marker="o",
            markersize=4,
        )
        plotted_any = True
    if plotted_any:
        ax.set_xlabel("Training step")
        ax.set_ylabel("Loss (streamed)")
        ax.set_title(f"GRPO training loss — streamed from job logs ({MODEL})")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / "loss_curve_local.png", dpi=140, bbox_inches="tight")
        plt.show()
    else:
        plt.close(fig)
        print("No loss values parsed yet.")

### Reward curves (all phases overlaid, smoothed)

In [ ]:
def rolling(series, window):
    return series.rolling(window=window, min_periods=1).mean()

# For each phase: aggregate to one mean reward per step BEFORE smoothing —
# without this, multi-completion runs and the early all-zero steps drag the
# line flat. Then overlay all phases on a single axis.
fig, ax = plt.subplots(figsize=(11, 5))
final_means = {}
for phase, df in dfs.items():
    per_step = (
        df.groupby("step", as_index=False)["reward"]
          .mean()
          .sort_values("step")
    )
    if per_step.empty:
        continue
    window = max(1, len(per_step) // 5)
    smoothed = rolling(per_step["reward"], window)
    ax.plot(
        per_step["step"],
        smoothed,
        label=phase,
        color=PHASE_COLORS.get(phase),
        linewidth=2.2,
        marker="o",
        markersize=4,
    )
    final_means[phase] = float(per_step["reward"].iloc[-1])

ax.set_xlabel("Training step")
ax.set_ylabel("Mean episode reward (smoothed)")
ax.set_title(f"GRPO reward curves — all phases ({MODEL})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "reward_curve.png", dpi=140, bbox_inches="tight")
plt.show()

print("\nFinal-step mean reward (paste into README scores table):")
for phase in PHASES:
    if phase in final_means:
        print(f"  {phase:>20s}: {final_means[phase]:.3f}")
    else:
        print(f"  {phase:>20s}: (no data)")

### Training loss curves (all phases overlaid)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plotted_any = False
for phase, log_history in log_histories.items():
    loss_records = [r for r in log_history if "loss" in r and "step" in r]
    if not loss_records:
        print(f"  [{phase}] no 'loss' entries in log_history — skipping")
        continue
    plotted_any = True
    steps = [r["step"] for r in loss_records]
    losses = [r["loss"] for r in loss_records]
    ax.plot(
        steps, losses,
        label=phase,
        color=PHASE_COLORS.get(phase),
        linewidth=2,
    )

if plotted_any:
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.set_title(f"GRPO training loss — all phases ({MODEL})")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "loss_curve.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    plt.close(fig)
    print("No 'loss' records in any phase's log_history — TRL may have logged with a different key.")

## 9. Cancel runaway / leftover jobs

Use this if:

- you Ctrl-C'd cell 11 mid-phase (the in-flight HF Job keeps running on
  HF until timeout or explicit cancel),
- you re-ran cell 11 after the *old* parallel-submit version had already
  fired off 3 jobs (the leftovers from that run will keep training in
  parallel with your new sequential run otherwise),
- you just want to stop a bad run early.

By default the cell below cancels **every** non-completed HF Job on your
account — flip `CANCEL_ONLY_THIS_RUN = True` to scope it to just the
phases submitted by the current kernel session.

In [ ]:
# Cancel any HF Job on your account that is still running / scheduled.
# Use this to nuke leftover jobs from a previous *parallel* submission of
# this notebook (before the cell 11 sequential fix), or after Ctrl-C'ing
# cell 11 mid-phase.
#
# Set CANCEL_ONLY_THIS_RUN = True to limit to phases submitted by the
# current kernel session (i.e. only entries in `JOBS`). Set to False to
# cancel every non-completed job on the whole account, which is what you
# want when cleaning up after the old "submit all 3 in parallel" version.
from huggingface_hub import HfApi, cancel_job

CANCEL_ONLY_THIS_RUN = False

api = HfApi(token=HF_TOKEN)
this_run_ids = set((JOBS or {}).values()) if "JOBS" in dir() else set()

# `list_jobs` is the modern endpoint; older hub versions call it `list_jobs`
# on HfApi too, so this works either way.
running_states = {"SCHEDULED", "RUNNING", "PENDING", "QUEUED"}
to_cancel = []
for j in api.list_jobs():
    status = (getattr(j, "status", None) or "").upper()
    if status not in running_states:
        continue
    if CANCEL_ONLY_THIS_RUN and j.id not in this_run_ids:
        continue
    to_cancel.append(j)

if not to_cancel:
    print("No running/scheduled jobs to cancel.")
else:
    for j in to_cancel:
        marker = " (this-run)" if j.id in this_run_ids else " (leftover)"
        print(f"cancelling {j.id}{marker}  status={getattr(j, 'status', '?')}")
        try:
            cancel_job(job_id=j.id)
        except Exception as exc:
            print(f"  cancel failed for {j.id}: {exc}")

## Done

You should now have three PNGs in `docs/plots/`, **rendered exclusively from
the `rewards.csv` and `log_history.json` produced by the three HF Jobs that
just finished above** — there is no fallback to older runs or to local CSV
files.

- `docs/plots/reward_curve.png` — reward curves (one line per phase).
- `docs/plots/loss_curve.png` — training loss (one line per phase).
- `docs/plots/reward_breakdown.png` — per-component breakdown (one subplot per phase).

The README references these PNG paths directly, so:

1. Commit `docs/plots/*.png` to surface the plots on the GitHub README.
2. Paste the printed final-step mean rewards into the
   "Scores from the latest run" table in `README.md` and
   `MetroCrowdManager/README.md`.
3. Re-run `scripts/push_env_to_hf_space.sh` (with `FORCE_README=1`) to refresh
   the HF Space landing page with the new numbers + plots.

Each fresh run of this notebook overwrites the same three filenames, so
re-running end-to-end is the canonical way to update the headline plots.

- **Live metrics**: <https://huggingface.co/spaces/DhiwakarDev/mcm-trackio>
- **Adapters + metrics (per phase)**:
  - <https://huggingface.co/DhiwakarDev/mcm-gemma3-27b-grpo-ticket_booking>
  - <https://huggingface.co/DhiwakarDev/mcm-gemma3-27b-grpo-ticket_issuance>
  - <https://huggingface.co/DhiwakarDev/mcm-gemma3-27b-grpo-crowd_announcement>
- **Env Space**: <https://huggingface.co/spaces/DhiwakarDev/openenv>